# 🗡️ Hey Aragorn Wake Word Training

Train a custom wake word model using micro-wake-word.

**Run each cell in order. Two required restarts — the notebook tells you when.**

## Step 0: Free Disk Space

In [ ]:
import shutil
for path in ['/usr/share/doc', '/usr/share/man', '/usr/share/locale',
             '/usr/lib/google-cloud-sdk', '/usr/local/android-sdk',
             '/usr/local/julia-1.9.4', '/content/sample_data']:
    shutil.rmtree(path, ignore_errors=True)
!apt-get clean -qq && apt-get autoremove -y -qq
!pip cache purge -q 2>/dev/null
print("Disk after cleanup:")
!df -h / | tail -1


## Step 1: Check GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout or 'nvidia-smi not found — make sure T4 GPU runtime is selected')


## Step 2a: Install All Packages

Run this first. When done, **restart once** (Runtime → Restart session), then continue from **Step 2b**.

In [ ]:
print("Installing espeak-ng...")
!apt-get install -y -q espeak-ng libespeak-ng-dev

print("\nInstalling piper-tts...")
!pip install piper-tts
!python -c "from piper import PiperVoice, SynthesisConfig; print('piper: OK')"

print("\nRemoving conflicting pre-installs...")
!pip uninstall -y jax jaxlib tensorstore tensorflow-decision-forests tensorflow-text opencv-python opencv-python-headless opencv-contrib-python shap ydf grain pytensor xarray-einstats rasterio tobler cupy-cuda12x 2>/dev/null

# TF 2.18 + CUDA bundle — required for CUDA 12.8 (the version Colab T4 now ships).
# TF 2.16.2 was built against CUDA 12.3 and cannot see the GPU on this runtime.
print("\nInstalling TensorFlow 2.18 (CUDA 12.8 compatible)...")
!pip install --quiet --timeout=300 'tensorflow[and-cuda]==2.18.0' protobuf==4.25.3 ml-dtypes==0.3.2

print("\nInstalling remaining deps...")
!pip install --quiet onnxruntime pyyaml datasets mmap-ninja tqdm audiomentations webrtcvad-wheels huggingface_hub

print("\n✅ Done! Restart: Runtime → Restart session, then continue from Step 2b.")


## Step 2b: Pin numpy & scipy

Only these two packages are reinstalled here to lock the correct versions. A **'Restart required'** popup will appear — **click it immediately**. Then continue from **Step 3**.

In [ ]:
!pip uninstall -y numpy scipy 2>/dev/null
!find /usr/local/lib/python3.*/dist-packages -name '*.pyc' -path '*/numpy/*' -delete 2>/dev/null
!find /usr/local/lib/python3.*/dist-packages -name '*.pyc' -path '*/scipy/*' -delete 2>/dev/null
!pip install --force-reinstall --no-cache-dir numpy==1.26.4 scipy==1.13.1
print("\n✅ numpy + scipy pinned. Click the Restart button now, then continue from Step 3.")


## ⚠️ Restart Required

**Runtime → Restart session** (Ctrl+M .)  →  continue from **Step 3**. Both restarts are now done.

## Step 3: Verify Installs & Clone Repositories

In [ ]:
import subprocess, os, sys, shutil

print(f"Python: {sys.version}")

import numpy as np
assert np.__version__ == '1.26.4', f"Bad numpy: {np.__version__}"
print(f"numpy {np.__version__}: OK")

import scipy; print(f"scipy {scipy.__version__}: OK")
import tensorflow as tf; print(f"tensorflow {tf.__version__}: OK")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU visible to TF: {gpus[0].name} ✅")
else:
    print("❌ TF cannot see GPU — check runtime type is T4 GPU")

r = subprocess.run([sys.executable, '-c',
    'from piper import PiperVoice, SynthesisConfig; print("piper: OK")'],
    capture_output=True, text=True)
print(r.stdout.strip() if r.returncode == 0 else f"piper FAILED:\n{r.stderr}")

# Always re-clone fresh — never leave a patched copy from a previous session
if os.path.exists('microWakeWord'):
    shutil.rmtree('microWakeWord')
subprocess.run(['git', 'clone',
    'https://github.com/kahrendt/microWakeWord.git'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e',
    'microWakeWord'], check=True)

# Patch validate_nonstreaming for Keras 3 compatibility.
# In Keras 3, model.evaluate() returns a list, not a dict.
# We wrap it to always return {metric_name: value} so train.py's
# dict indexing works correctly and validation metrics are non-zero.
CUT_MARKER = '\n# === NOTEBOOK PATCH ==='
train_py = 'microWakeWord/microwakeword/train.py'
with open(train_py, 'r') as f:
    src = f.read()
if CUT_MARKER in src:
    src = src[:src.index(CUT_MARKER)]

patch = CUT_MARKER + '''

_orig_validate_nonstreaming = validate_nonstreaming

def validate_nonstreaming(config, data_processor, model, test_set):
    _real_evaluate = model.evaluate
    def _evaluate_as_dict(*a, **kw):
        kw.setdefault('return_dict', True)
        result = _real_evaluate(*a, **kw)
        if isinstance(result, (list, tuple)):
            names = [m.name for m in model.metrics]
            result = dict(zip(names, result))
        return result
    model.evaluate = _evaluate_as_dict
    try:
        return _orig_validate_nonstreaming(config, data_processor, model, test_set)
    finally:
        model.evaluate = _real_evaluate
'''

with open(train_py, 'w') as f:
    f.write(src + patch)
print("microWakeWord cloned + patched ✅")

if not os.path.exists('piper-sample-generator'):
    subprocess.run(['git', 'clone',
        'https://github.com/rhasspy/piper-sample-generator.git'], check=True)
print("piper-sample-generator: OK")
print("\n✅ All verified and ready!")


## Step 4: Download Piper Voice Model

In [ ]:
import os, urllib.request
os.makedirs('piper-sample-generator/models', exist_ok=True)
url  = ('https://github.com/rhasspy/piper-sample-generator/releases/'
        'download/v2.0.0/en_US-libritts_r-medium.pt')
path = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'
if not os.path.exists(path):
    print('Downloading...')
    urllib.request.urlretrieve(url, path)
    print('✅ Done!')
else:
    print('✅ Already exists')


## Step 5: Configure

Adjust `TARGET_WORD` until Step 6 sounds right.

Tips: underscores between syllables (`hey_air_uh_gorn`), `sh`/`ch`/`th` for those sounds, `ee`/`oo` for long vowels.

In [ ]:
TARGET_WORD    = 'hey_air_uh_gorn'
NUM_SAMPLES    = 1000
TRAINING_STEPS = 10000
print(f'Word:    {TARGET_WORD}')
print(f'Samples: {NUM_SAMPLES}')
print(f'Steps:   {TRAINING_STEPS}')


## Step 6: Generate Test Sample

In [ ]:
import subprocess, os, sys
from IPython.display import Audio, display
os.makedirs('generated_samples', exist_ok=True)
env = {**os.environ, 'PYTHONPATH': os.path.abspath('piper-sample-generator')}
cmd = [sys.executable, '-m', 'piper_sample_generator',
       TARGET_WORD, '--max-samples', '1', '--batch-size', '1',
       '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
       '--output-dir', 'generated_samples']
result = subprocess.run(cmd, capture_output=True, text=True, env=env)
if result.returncode == 0:
    wavs = sorted([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
    if wavs:
        print(f'Phonetic spelling: "{TARGET_WORD}"')
        print('Adjust TARGET_WORD in Step 5 if needed.\n')
        display(Audio(os.path.join('generated_samples', wavs[0])))
else:
    print('STDOUT:', result.stdout); print('STDERR:', result.stderr)


## Step 7: Generate All Samples

In [ ]:
import subprocess, os, sys
print(f'Generating {NUM_SAMPLES} samples...')
env = {**os.environ, 'PYTHONPATH': os.path.abspath('piper-sample-generator')}
cmd = [sys.executable, '-m', 'piper_sample_generator',
       TARGET_WORD, '--max-samples', str(NUM_SAMPLES), '--batch-size', '100',
       '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
       '--output-dir', 'generated_samples']
result = subprocess.run(cmd, capture_output=True, text=True, env=env)
if result.returncode == 0:
    count = len([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
    print(f'✅ Generated {count} samples!')
else:
    print('STDOUT:', result.stdout); print('STDERR:', result.stderr)


## Step 8: Download Augmentation Data

In [ ]:
import os
os.makedirs('mit_rirs', exist_ok=True)
if not os.listdir('mit_rirs'):
    print('Downloading RIRs + background noises (~1.3 GB)...')
    !wget --progress=bar:force -O /tmp/rirs_noises.zip https://www.openslr.org/resources/28/rirs_noises.zip
    !unzip -q /tmp/rirs_noises.zip -d mit_rirs
    print('Extracted!')
else:
    print('Already downloaded')
noise_dir = 'mit_rirs/RIRS_NOISES/pointsource_noises'
if os.path.exists(noise_dir):
    n = len([f for f in os.listdir(noise_dir) if f.endswith('.wav')])
    print(f'✅ {n} background noise files ready')
else:
    print('❌ pointsource_noises not found')


## 🧹 Disk Cleanup (After Step 8)

In [ ]:
import os, subprocess, sys
if os.path.exists('/tmp/rirs_noises.zip'):
    os.remove('/tmp/rirs_noises.zip'); print('Deleted rirs_noises.zip')
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], capture_output=True)
print('pip cache purged')
!df -h / | tail -1


## Step 9: Generate Spectrograms

In [ ]:
import os, sys
if 'microWakeWord' not in sys.path:
    sys.path.insert(0, 'microWakeWord')

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

# Two mmaps: training/ and testing/
# The evaluator needs positive test samples or it hits ZeroDivisionError.
MMAP_TRAIN = 'generated_augmented_features/training/wakeword_mmap'
MMAP_TEST  = 'generated_augmented_features/testing/wakeword_mmap'
os.makedirs(os.path.dirname(MMAP_TRAIN), exist_ok=True)
os.makedirs(os.path.dirname(MMAP_TEST),  exist_ok=True)

clips = Clips(input_directory='generated_samples', file_pattern='*.wav',
    max_clip_duration_s=None, remove_silence=False,
    random_split_seed=10, split_count=0.1)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1, 'TanhDistortion': 0.1,
        'PitchShift': 0.1, 'BandStopFilter': 0.1,
        'AddColorNoise': 0.1, 'AddBackgroundNoise': 0.75,
        'Gain': 1.0, 'RIR': 0.5,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['mit_rirs/RIRS_NOISES/pointsource_noises'],
    background_min_snr_db=-5, background_max_snr_db=10,
    min_jitter_s=0.195, max_jitter_s=0.205,
)

spectrograms = SpectrogramGeneration(
    clips=clips, augmenter=augmenter, slide_frames=10, step_ms=10)

print('Writing train split...')
RaggedMmap.from_generator(
    out_dir=MMAP_TRAIN,
    sample_generator=spectrograms.spectrogram_generator(split='train', repeat=2),
    batch_size=100, verbose=True)

print('Writing test split...')
RaggedMmap.from_generator(
    out_dir=MMAP_TEST,
    sample_generator=spectrograms.spectrogram_generator(split='test', repeat=1),
    batch_size=100, verbose=True)

mmap_tr = RaggedMmap(MMAP_TRAIN)
mmap_te = RaggedMmap(MMAP_TEST)
assert len(mmap_tr) > 0, "train mmap is empty — check generated_samples/ has .wav files."
assert len(mmap_te) > 0, "test mmap is empty — split_count=0.1 yielded 0 test clips; increase NUM_SAMPLES."
print(f'✅ {len(mmap_tr)} train spectrograms | {len(mmap_te)} test spectrograms')
print(f'   Shape of first: {mmap_tr[0].shape}')


## Step 10: Download Negative Datasets

In [ ]:
import os, zipfile
from huggingface_hub import hf_hub_download, list_repo_files
os.makedirs('negative_datasets', exist_ok=True)
repo_id, repo_type = 'kahrendt/microwakeword', 'dataset'
zip_files = [f for f in list_repo_files(repo_id, repo_type=repo_type) if f.endswith('.zip')]
print(f'Found: {zip_files}')
for fname in zip_files:
    base = os.path.splitext(os.path.basename(fname))[0]
    if not os.path.exists(f'negative_datasets/{base}'):
        print(f'Downloading {fname}...')
        local = hf_hub_download(repo_id=repo_id, filename=fname, repo_type=repo_type)
        with zipfile.ZipFile(local, 'r') as zf:
            zf.extractall('negative_datasets')
        print(f'  done: {base}')
    else:
        print(f'  exists: {base}')
neg_dirs = [d for d in os.listdir('negative_datasets')
            if os.path.isdir(f'negative_datasets/{d}')]
print(f'\n✅ Directories: {neg_dirs}')


## 🧹 Disk Cleanup (After Step 10)

In [ ]:
import os, shutil
hf_cache = os.path.expanduser('~/.cache/huggingface')
if os.path.exists(hf_cache):
    size = sum(os.path.getsize(os.path.join(dp,f))
               for dp,_,fs in os.walk(hf_cache) for f in fs)
    shutil.rmtree(hf_cache)
    print(f'Deleted HuggingFace cache ({size/1e9:.1f} GB freed)')
else:
    print('No HuggingFace cache found')
!df -h / | tail -1


## Step 11: Create Training Config

In [ ]:
import yaml, os

neg_dirs   = sorted([d for d in os.listdir('negative_datasets')
                     if os.path.isdir(f'negative_datasets/{d}')
                     and not d.startswith('__')])
eval_dirs  = [d for d in neg_dirs if 'eval' in d]
train_dirs = [d for d in neg_dirs if 'eval' not in d]
print(f'Train negatives : {train_dirs}')
print(f'Eval  negatives : {eval_dirs}')

neg_features = []
for d in train_dirs:
    neg_features.append({'features_dir': f'negative_datasets/{d}',
        'sampling_weight': 10.0, 'penalty_weight': 1.0,
        'truth': False, 'truncation_strategy': 'random', 'type': 'mmap'})
for d in eval_dirs:
    neg_features.append({'features_dir': f'negative_datasets/{d}',
        'sampling_weight': 0.0, 'penalty_weight': 1.0,
        'truth': False, 'truncation_strategy': 'split', 'type': 'mmap'})

# Two positive entries:
#   train — used during training (sampling_weight > 0)
#   test  — eval only, never sampled during training (sampling_weight = 0)
pos_feature_train = {
    'features_dir': 'generated_augmented_features',
    'sampling_weight': 2.0, 'penalty_weight': 1.0,
    'truth': True, 'truncation_strategy': 'truncate_start', 'type': 'mmap'}

pos_feature_test = {
    'features_dir': 'generated_augmented_features',
    'sampling_weight': 0.0, 'penalty_weight': 1.0,
    'truth': True, 'truncation_strategy': 'split', 'type': 'mmap'}

config = {
    'window_step_ms': 10,
    'train_dir': 'trained_models/wakeword',
    'spectrogram_length': 204,
    'stride': 3,
    'features': [pos_feature_train, pos_feature_test] + neg_features,
    'training_steps': [TRAINING_STEPS],
    'positive_class_weight': [1],
    'negative_class_weight': [20],
    'learning_rates': [0.001],
    'batch_size': 128,
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
    'target_minimization': 0.9,
    'minimization_metric': '',
    'maximization_metric': 'average_viable_recall',
}

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print('✅ Config saved.')
print(f'   Positive train mmap : generated_augmented_features/training/wakeword_mmap')
print(f'   Positive test  mmap : generated_augmented_features/testing/wakeword_mmap')


## Step 12: Train Model

This takes 1–2 hours on a T4. GPU RAM should spike within the first minute.

In [ ]:
import sys, os, shutil

# Confirm GPU is visible before starting
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'✅ GPU: {gpus[0].name}')
else:
    raise RuntimeError('No GPU detected — go to Runtime → Change runtime type → T4 GPU, then re-run from Step 3.')

if os.path.exists('trained_models/wakeword'):
    shutil.rmtree('trained_models/wakeword')
    print('Cleared previous trained_models/wakeword')

print('Starting training...')
print(f'~{TRAINING_STEPS // 10000} hour(s)...')
print('Steps print every 500 iterations.')
print('-' * 60)

# Run training in-process so TF uses the GPU already claimed above.
# absl registers flags at module import time — importing twice causes a
# DuplicateFlagError crash. The fix: import once, parse flags manually,
# then call the trainer's main() function directly.
import absl.logging
import absl.flags
import absl.app

# Route absl logs to Python logging so Colab shows them live
absl.logging.set_verbosity(absl.logging.INFO)
absl.logging.use_python_logging()

# Build the argument list the trainer expects
train_args = [
    '--training_config=training_parameters.yaml',
    '--train=1',
    '--restore_checkpoint=0',
    '--test_tf_nonstreaming=0',
    '--test_tflite_nonstreaming=0',
    '--test_tflite_nonstreaming_quantized=0',
    '--test_tflite_streaming=0',
    '--test_tflite_streaming_quantized=1',
    'mixednet',
    '--pointwise_filters=64,64,64,64',
    '--repeat_in_block=1,1,1,1',
    '--mixconv_kernel_sizes=[5],[7,11],[9,15],[23]',
    '--residual_connection=0,0,0,0',
    '--first_conv_filters=32',
    '--first_conv_kernel_size=5',
    '--stride=3',
]

# Import the trainer module (registers all absl flags as a side effect)
import microwakeword.model_train_eval as _mte

# Parse the flags into absl's global FLAGS object
_remaining = absl.flags.FLAGS(train_args, known_only=True)

# Call main() directly — bypasses absl.app.run() which calls sys.exit
try:
    _mte.main(_remaining)
    print('-' * 60)
    print('✅ Training complete!')
except SystemExit as e:
    if e.code in (0, None):
        print('-' * 60)
        print('✅ Training complete!')
    else:
        print('-' * 60)
        print(f'❌ Training exited with code {e.code}')
except Exception as e:
    print('-' * 60)
    print(f'❌ Training raised exception: {type(e).__name__}: {e}')
    raise


## Step 13: Download Model

In [ ]:
import os
from google.colab import files
model_path = ('trained_models/wakeword/'
              'tflite_stream_state_internal_quant/'
              'stream_state_internal_quant.tflite')
if os.path.exists(model_path):
    print(f'✅ Model: {os.path.getsize(model_path)/1024:.1f} KB')
    files.download(model_path)
    print('\n🎉 Done!')
else:
    print('❌ Model not found — check training output above')
